In [ ]:
# DR baseline v3: APTOS + DDR (many cameras), strong camera augmentation; external test on IDRiD (never seen).
import glob, os, tarfile
import pandas as pd
_ = [tarfile.open(b, "r:xz").extractall("/kaggle/working") for b in sorted(glob.glob("/kaggle/input/**/dr_*_code.bin", recursive=True))]
os.chdir("/kaggle/working/dr-baseline")
APTOS_DIR = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.csv", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "train_images"))][0]
DDR_DIR = os.path.dirname(glob.glob("/kaggle/input/**/DR_grading", recursive=True)[0])
GT_TRAIN = glob.glob("/kaggle/input/**/2. Groundtruths/a.*.csv", recursive=True)[0]
GT_TEST = glob.glob("/kaggle/input/**/2. Groundtruths/b.*.csv", recursive=True)[0]
IMG_TRAIN = glob.glob("/kaggle/input/**/1. Original Images/a. Training Set", recursive=True)[0]
IMG_TEST = glob.glob("/kaggle/input/**/1. Original Images/b. Testing Set", recursive=True)[0]
for part, gt in (("train", GT_TRAIN), ("test", GT_TEST)):
    lab = pd.read_csv(gt).iloc[:, :3]
    lab.columns = ["Image name", "Retinopathy grade", "Risk of macular edema"]
    lab.dropna().to_csv(f"/kaggle/working/idrid_{part}.csv", index=False)
print("APTOS:", APTOS_DIR, "| DDR:", DDR_DIR, "| files:", sorted(os.listdir(".")))

In [ ]:
!python prepare_data.py train-sets --aptos-dir {APTOS_DIR} --ddr-dir {DDR_DIR} --out-dir data/processed

In [ ]:
!python train.py --manifest data/processed/manifest.csv --out runs/v3 --aug strong --epochs 8

In [ ]:
!python evaluate.py --run-dir runs/v3 --csv data/processed/manifest.csv --split test

In [ ]:
!python prepare_data.py external --name idrid_train --csv /kaggle/working/idrid_train.csv --image-dir "{IMG_TRAIN}" --image-col "Image name" --label-col "Retinopathy grade" --dme-col "Risk of macular edema" --out-dir data/processed
!python prepare_data.py external --name idrid_test --csv /kaggle/working/idrid_test.csv --image-dir "{IMG_TEST}" --image-col "Image name" --label-col "Retinopathy grade" --dme-col "Risk of macular edema" --out-dir data/processed
pd.concat([pd.read_csv("data/processed/external_idrid_train.csv"), pd.read_csv("data/processed/external_idrid_test.csv")]).to_csv("data/processed/external_idrid_all.csv", index=False)
!python evaluate.py --run-dir runs/v3 --csv data/processed/external_idrid_all.csv

In [ ]:
# Site tuning, done fairly: threshold from IDRiD's training folder, measured on IDRiD's testing folder
!python evaluate.py --run-dir runs/v3 --csv data/processed/external_idrid_train.csv --n-boot 0
!python site_threshold.py --predictions runs/v3/eval_external_idrid_train/predictions.csv --thresholds runs/v3/thresholds.json --out runs/v3/thresholds_idrid.json
!python evaluate.py --run-dir runs/v3 --csv data/processed/external_idrid_test.csv --thresholds runs/v3/thresholds_idrid.json --name idrid_test_site_tuned

In [ ]:
!python rim_test.py --run-dir runs/v3 --csv data/processed/external_idrid_all.csv --split external --out runs/v3/rim_test_idrid

In [ ]:
!rm -rf data/processed/images runs/v3/last.pt; du -sh /kaggle/working; find /kaggle/working -type f | wc -l